In [4]:
import sys
from pathlib import Path
import pandas as pd
# Make src/ importable
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT = Path("..").resolve()  # assuming notebook is in /notebooks
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", DATA_PROCESSED)


Project root: C:\Users\User\Documents\retailmind-prototype
Processed data dir: C:\Users\User\Documents\retailmind-prototype\data\processed


In [ ]:
def enrich_with_satisfaction_flags(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    """
    Clean satisfaction_mean and add a low_satisfaction flag.

    Rules:
    - satisfaction_mean must be numeric
    - low_satisfaction is True when:
        * speaker == 'USER'
        * satisfaction_mean < 3
        * is_overall == False  
    """
    df = df.copy()
    df["dataset"] = dataset_name

    # Ensure numeric satisfaction
    df["satisfaction_mean"] = pd.to_numeric(df["satisfaction_mean"], errors="coerce")


    # Low-satisfaction flag (user expressing dissatisfaction at turn level)
    df["low_satisfaction"] = (
        (df["speaker"] == "USER")
        & (df["satisfaction_mean"] < 3)
        & (df["is_overall"] == False)
    )

    # Simple summary
    print(f"[{dataset_name}] Total rows: {len(df)}")
    print(f"[{dataset_name}] Low-satisfaction rows: {df['low_satisfaction'].sum()}")
    print(f"[{dataset_name}] Low-satisfaction ratio: {df['low_satisfaction'].mean():.3f}")

    return df


In [6]:
# Load SGD processed turns
sgd_path = DATA_PROCESSED / "sgd_turns.parquet"
df_sgd = pd.read_parquet(sgd_path)

print("SGD columns:", df_sgd.columns.tolist())
df_sgd.head()


SGD columns: ['conv_id', 'turn_id', 'speaker', 'text', 'dialog_act', 'scores_raw', 'scores', 'is_overall', 'satisfaction_mean', 'label']


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label
0,0,1,USER,What is the weather like on the March 4th?,INFORM,"3,3,3","[3, 3, 3]",False,3.000000,INFORM
1,0,2,SYSTEM,In which city should I look?,REQUEST,None,None,False,NaN,REQUEST
2,0,3,USER,The weather in Mill Valley.,INFORM,"4,3,3","[4, 3, 3]",False,3.333333,INFORM
3,0,4,SYSTEM,The weather should be around 90 degrees and th...,OFFER,None,None,False,NaN,OFFER
4,0,5,USER,How humid will the temperature be?,REQUEST,"3,3,3","[3, 3, 3]",False,3.000000,REQUEST


In [7]:
# Enrich SGD with low_satisfaction flag
df_sgd_enriched = enrich_with_satisfaction_flags(df_sgd, dataset_name="SGD")


[SGD] Total rows: 26666
[SGD] Low-satisfaction rows: 2494
[SGD] Low-satisfaction ratio: 0.094


In [8]:
df_sgd_enriched.head()

,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label,dataset,low_satisfaction
0,0,1,USER,What is the weather like on the March 4th?,INFORM,"3,3,3","[3, 3, 3]",False,3.000000,INFORM,SGD,False
1,0,2,SYSTEM,In which city should I look?,REQUEST,None,None,False,NaN,REQUEST,SGD,False
2,0,3,USER,The weather in Mill Valley.,INFORM,"4,3,3","[4, 3, 3]",False,3.333333,INFORM,SGD,False
3,0,4,SYSTEM,The weather should be around 90 degrees and th...,OFFER,None,None,False,NaN,OFFER,SGD,False
4,0,5,USER,How humid will the temperature be?,REQUEST,"3,3,3","[3, 3, 3]",False,3.000000,REQUEST,SGD,False


In [9]:
# Save enriched SGD parquet in data/processed
sgd_out_path = DATA_PROCESSED / "sgd_turns_satisfaction.parquet"
df_sgd_enriched.to_parquet(sgd_out_path, index=False)
print("Saved:", sgd_out_path)


Saved: C:\Users\User\Documents\retailmind-prototype\data\processed\sgd_turns_satisfaction.parquet


In [10]:
# Load CCPE processed turns
ccpe_path = DATA_PROCESSED / "ccpe_turns.parquet"
df_ccpe = pd.read_parquet(ccpe_path)

print("CCPE columns:", df_ccpe.columns.tolist())
df_ccpe.head()


CCPE columns: ['conv_id', 'turn_id', 'speaker', 'text', 'entity_tag', 'scores_raw', 'scores', 'is_overall', 'satisfaction_mean', 'label']


,conv_id,turn_id,speaker,text,entity_tag,scores_raw,scores,is_overall,satisfaction_mean,label
0,0,1,SYSTEM,Do you like movies like Thor?,ENTITY_NAME+MOVIE_OR_SERIES,None,None,False,NaN,ENTITY_NAME+MOVIE_OR_SERIES
1,0,2,USER,"No, I don't like Thor.",ENTITY_NAME+MOVIE_OR_SERIES,"3,2,2","[3, 2, 2]",False,2.333333,ENTITY_NAME+MOVIE_OR_SERIES
2,0,3,SYSTEM,Ok. What is it about this type of movie that y...,OTHER,None,None,False,NaN,OTHER
3,0,4,USER,I don't like all the,OTHER,"3,2,2","[3, 2, 2]",False,2.333333,OTHER
4,0,5,USER,Like the I don't know. Like is it voice acting?,OTHER,"3,2,2","[3, 2, 2]",False,2.333333,OTHER


In [11]:
# Enrich CCPE with low_satisfaction flag
df_ccpe_enriched = enrich_with_satisfaction_flags(df_ccpe, dataset_name="CCPE")


[CCPE] Total rows: 12436
[CCPE] Low-satisfaction rows: 1595
[CCPE] Low-satisfaction ratio: 0.128


In [12]:
# Save enriched CCPE parquet in data/processed
ccpe_out_path = DATA_PROCESSED / "ccpe_turns_satisfaction.parquet"
df_ccpe_enriched.to_parquet(ccpe_out_path, index=False)
print("Saved:", ccpe_out_path)


Saved: C:\Users\User\Documents\retailmind-prototype\data\processed\ccpe_turns_satisfaction.parquet


In [13]:
# Combine enriched English datasets (SGD + CCPE; for now)
df_english = pd.concat(
    [df_sgd_enriched, df_ccpe_enriched],
    ignore_index=True
)

print("Combined English rows:", len(df_english))
print("Combined low-satisfaction rows:", df_english["low_satisfaction"].sum())

english_out_path = DATA_PROCESSED / "uss_english_turns_satisfaction.parquet"
df_english.to_parquet(english_out_path, index=False)
print("Saved:", english_out_path)


Combined English rows: 39102
Combined low-satisfaction rows: 4089
Saved: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_turns_satisfaction.parquet


In [14]:
# Sanity check: distribution of satisfaction_mean and low_satisfaction
df_english["satisfaction_mean"].describe()


count    20693.000000
mean         3.098643
std          0.368407
min          1.200000
25%          3.000000
50%          3.000000
75%          3.333333
max          5.000000
Name: satisfaction_mean, dtype: float64

In [15]:
df_english["low_satisfaction"].value_counts(normalize=True)

low_satisfaction
False    0.895427
True     0.104573
Name: proportion, dtype: float64